In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point, MultiLineString, LineString
# import pygmt
import networkx as nx
from tqdm import tqdm, trange
%matplotlib qt

In [3]:
right_bank = gpd.read_file('/Users/zoltan/Dropbox/Channels/Fluvial/Mamore_cutoff/new_GIS_data/mamore_right_bank.shp')

In [4]:
right_bank

,migr_rate,direction,age,area,geometry
0,19.841002,1,0,812.806173,"POLYGON ((280904.628 -1669530.555, 280891.669 ..."
1,10.012592,1,0,405.830051,"POLYGON ((280940.756 -1669513.309, 280934.657 ..."
2,4.554054,1,0,181.183133,"POLYGON ((280977.269 -1669496.886, 280975.828 ..."
3,2.035753,1,0,81.182172,"POLYGON ((281014.475 -1669482.193, 281013.668 ..."
4,-2.365115,-1,0,82.023968,"MULTIPOLYGON (((281052.831 -1669470.700, 28105..."
...,...,...,...,...,...
27537,-66.220772,-1,29,2482.725835,"POLYGON ((285066.631 -1650294.599, 285029.743 ..."
27538,-63.458496,-1,29,2092.249600,"POLYGON ((285029.743 -1650303.841, 284996.749 ..."
27539,-48.919154,-1,29,3186.100446,"POLYGON ((284996.749 -1650312.131, 284929.242 ..."
27540,-28.473945,-1,29,1231.653896,"POLYGON ((284929.242 -1650315.450, 284886.303 ..."


In [5]:
poly_graph_1 = nx.read_gpickle("/Users/zoltan/Dropbox/Channels/Fluvial/Mamore_cutoff/new_GIS_data/poly_graph_1.gpickle")

In [6]:
poly_graph_2 = nx.read_gpickle("/Users/zoltan/Dropbox/Channels/Fluvial/Mamore_cutoff/new_GIS_data/poly_graph_2.gpickle")

In [7]:
import matplotlib as mpl
fig = plt.figure()
ax = fig.add_subplot(111)
norm = mpl.colors.Normalize(vmin=-100, vmax=100)
m = mpl.cm.ScalarMappable(norm=norm, cmap='coolwarm')
for node in tqdm(poly_graph_1.nodes):
        poly = poly_graph_1.nodes[node]['poly']
        if type(poly) == Polygon:
            plt.fill(poly.exterior.xy[0], poly.exterior.xy[1], 
                facecolor = m.to_rgba(poly_graph_1.nodes[node]['migr_rate']*poly_graph_1.nodes[node]['direction']), 
                edgecolor='k', linewidth=0.25, alpha=0.4)
plt.axis('equal')

100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 27542/27542 [00:07<00:00, 3845.47it/s]


(280424.35791427176,
 287652.17915516545,
 -1671170.212414464,
 -1649023.0974323864)

In [10]:
def load_landsat_image(img_folder, bands):
    image = {} # create an empty dictionary
    path = Path(img_folder)
    for band in bands:
        # landsat images end with *_RT_B#.TIF
        file = next(path.glob(f'*_T1_{band}.TIF')) # filename
        print(f'Opening file {file}')
        ds = rasterio.open(file) # open file using rasterio
        image.update({band: ds.read(1)}) # store band in dictionary
    return image, ds.transform, ds.bounds
    
def normalized_difference(img, b1, b2, eps=0.0001):
    band1 = np.where((img[b1]==0) & (img[b2]==0), np.nan, img[b1])
    band2 = np.where((img[b1]==0) & (img[b2]==0), np.nan, img[b2])
    return (band1 - band2) / (band1 + band2)

In [11]:
from pathlib import Path
import rasterio
folder = '/Users/zoltan/Dropbox/P4GR/RTX_2023/landsat_data_2018/'
img, transform, bounds = load_landsat_image(folder, ['B2', 'B3', 'B4', 'B5', 'B6', 'B7'])

Opening file /Users/zoltan/Dropbox/P4GR/RTX_2023/landsat_data_2018/LC08_L1TP_232070_20180622_20200831_02_T1_B2.TIF
Opening file /Users/zoltan/Dropbox/P4GR/RTX_2023/landsat_data_2018/LC08_L1TP_232070_20180622_20200831_02_T1_B3.TIF
Opening file /Users/zoltan/Dropbox/P4GR/RTX_2023/landsat_data_2018/LC08_L1TP_232070_20180622_20200831_02_T1_B4.TIF
Opening file /Users/zoltan/Dropbox/P4GR/RTX_2023/landsat_data_2018/LC08_L1TP_232070_20180622_20200831_02_T1_B5.TIF
Opening file /Users/zoltan/Dropbox/P4GR/RTX_2023/landsat_data_2018/LC08_L1TP_232070_20180622_20200831_02_T1_B6.TIF
Opening file /Users/zoltan/Dropbox/P4GR/RTX_2023/landsat_data_2018/LC08_L1TP_232070_20180622_20200831_02_T1_B7.TIF


In [12]:
xmin = (transform * (2000, 3500))[0]
ymax = (transform * (2000, 3500))[1]
xmax = (transform * (3000, 4500))[0]
ymin = (transform * (3000, 4500))[1]

rgb2 = np.stack([img['B7'], img['B5'], img['B3']], axis=-1)
rgb2 = rgb2/rgb2.max() * 2.5
fig = plt.figure(figsize = (10, 10))
ax = fig.add_subplot(111)
ax.imshow(rgb2[3500:4500, 2000:3000, :], extent = [xmin, xmax, ymin, ymax])

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).


In [13]:
norm = mpl.colors.Normalize(vmin=-200, vmax=200)
m = mpl.cm.ScalarMappable(norm=norm, cmap='coolwarm')
plt.figure()
plt.imshow(rgb2, extent = [bounds[0], bounds[2], bounds[1], bounds[3]])
for node in tqdm(poly_graph_1.nodes):
        poly = poly_graph_1.nodes[node]['poly']
        if type(poly) == Polygon:
            plt.fill(poly.exterior.xy[0], poly.exterior.xy[1], 
                facecolor = m.to_rgba(poly_graph_1.nodes[node]['migr_rate']*poly_graph_1.nodes[node]['direction']), 
                edgecolor='k', linewidth=0.25, alpha=1)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 27542/27542 [00:08<00:00, 3155.99it/s]


In [55]:
ylim = plt.gca().get_ylim()
xlim = plt.gca().get_xlim()

In [62]:
xlim

(275972.0840789568, 291897.266767988)

In [63]:
ylim

(-1671482.6104035194, -1647641.2654740224)

In [59]:
norm = mpl.colors.Normalize(vmin=-200, vmax=200)
m = mpl.cm.ScalarMappable(norm=norm, cmap='coolwarm_r')
plt.figure()
plt.imshow(rgb2, extent = [bounds[0], bounds[2], bounds[1], bounds[3]])
for node in tqdm(poly_graph_2.nodes):
        poly = poly_graph_2.nodes[node]['poly']
        if type(poly) == Polygon:
            plt.fill(poly.exterior.xy[0], poly.exterior.xy[1], 
                facecolor = m.to_rgba(poly_graph_2.nodes[node]['migr_rate']*poly_graph_2.nodes[node]['direction']), 
                edgecolor='k', linewidth=0.25, alpha=1)
plt.xlim(xlim)
plt.ylim(ylim)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
100%|██████████████████████████████████████████████████████████████████████████████████████| 28123/28123 [00:09<00:00, 2888.41it/s]


(-1671482.6104035194, -1647641.2654740224)

2023-06-28 13:00:39.725 python3.11[99789:4892853] +[CATransaction synchronize] called within transaction
2023-06-28 13:00:46.941 python3.11[99789:4892853] +[CATransaction synchronize] called within transaction
2023-06-28 13:00:48.086 python3.11[99789:4892853] +[CATransaction synchronize] called within transaction
2023-06-28 13:00:48.354 python3.11[99789:4892853] +[CATransaction synchronize] called within transaction


In [61]:
a = np.array([[-200, 200]])
plt.figure(figsize=(9, 1.5))
img = plt.imshow(a, cmap="coolwarm_r")
plt.gca().set_visible(False)
cax = plt.axes([0.1, 0.2, 0.8, 0.6])
plt.colorbar(orientation="horizontal", cax=cax)
# pl.savefig("colorbar.pdf")

2023-06-28 13:26:22.975 python3.11[99789:4892853] +[CATransaction synchronize] called within transaction
2023-06-28 13:26:30.678 python3.11[99789:4892853] +[CATransaction synchronize] called within transaction
2023-06-28 13:26:31.679 python3.11[99789:4892853] +[CATransaction synchronize] called within transaction
2023-06-28 13:26:31.945 python3.11[99789:4892853] +[CATransaction synchronize] called within transaction
